# 04c — XAI: Reproducible LIME vs SHAP Pipeline on IndoBERT Sentiment Model

Notebook ini berisi pipeline evaluasi interpretabilitas (LIME vs SHAP) secara kuantitatif dan kualitatif untuk model sentimen biner (Negatif vs Positif) IndoBERT.

### Target Penelitian Bab IV:
1. **Reproducible XAI Pipeline**: Menggunakan wrapper terstandar untuk memfasilitasi audit explainability.
2. **Analisis Kualitatif (Local)**: Visualisasi horizontal bar chart berdampingan untuk 4 sampel representatif (2 benar, 2 misklasifikasi).
3. **Analisis Kuantitatif (Global)**:
   - Konsistensi fitur: **Jaccard Similarity** (tumpang tindih kata) & **Spearman Rank Correlation** (korelasi bobot).
   - Keandalan model: **Comprehensiveness (AOPC)** & **Sufficiency**.
   - Efisiensi: Perbandingan **Runtime** komputasi.
4. **Export Artefak Skripsi**: Gambar disimpan di `outputs/figures/xai/` dan tabel disimpan di `outputs/finetuning_indobert/reports/`.

In [1]:
# 0. Setup Google Colab (skip jika dijalankan secara lokal)
import sys, os
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Google Colab terdeteksi — melakukan mounting Google Drive...")
    from google.colab import drive
    drive.mount("/content/drive")

    # Sesuaikan dengan path folder proyek Anda di Drive
    PROJECT_PATH = "/content/drive/MyDrive/xai_lime_vs_shap"
    if os.path.isdir(PROJECT_PATH):
        os.chdir(PROJECT_PATH)
        print(f"Direktori kerja aktif: {os.getcwd()}")
    else:
        print(f"WARNING: Direktori '{PROJECT_PATH}' tidak ditemukan.")

    # Install library XAI
    !pip install -q transformers torch datasets scikit-learn lime shap scipy matplotlib seaborn
else:
    print(f"Menjalankan secara lokal. Direktori kerja: {os.getcwd()}")

Google Colab terdeteksi — melakukan mounting Google Drive...
Mounted at /content/drive
Direktori kerja aktif: /content/drive/MyDrive/xai_lime_vs_shap
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 6.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
# 1. Import Library & Setup Environment
import warnings
import pandas as pd
import numpy as np
import time
import torch
import shap
from lime.lime_text import LimeTextExplainer

# Tambahkan path src agar modul proyek terdeteksi
sys.path.append(os.path.abspath(".."))
if os.path.abspath(".") not in sys.path:
    sys.path.append(os.path.abspath("."))

from src.xai.explainer import PredictProbaWrapper, load_sentiment_model_and_tokenizer
from src.xai.metrics import (
    extract_lime_features, extract_shap_features,
    calculate_jaccard_similarity, calculate_spearman_correlation,
    remove_tokens, keep_only_tokens,
    calculate_comprehensiveness, calculate_sufficiency
)
from src.xai.visualizer import plot_local_comparison, plot_metric_distributions, plot_aopc_curves

warnings.filterwarnings("ignore")
print("Modul XAI dan visualisasi berhasil dimuat.")

Modul XAI dan visualisasi berhasil dimuat.


In [3]:
# 2. Konfigurasi Path Proyek
PROJECT_ROOT = Path(os.getcwd())
MODEL_DIR = PROJECT_ROOT / "outputs" / "finetuning_indobert" / "model" / "final_indobert_sentiment"
TOKENIZER_DIR = PROJECT_ROOT / "outputs" / "finetuning_indobert" / "tokenizer" / "indobert_tokenizer"
TEST_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "tokopedia_reviews_binary_test.csv"
TRAIN_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "tokopedia_reviews_binary_train.csv"
FIG_DIR = PROJECT_ROOT / "outputs" / "figures" / "xai"
REPORTS_DIR = PROJECT_ROOT / "outputs" / "finetuning_indobert" / "reports"

FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Model Path      : {MODEL_DIR}")
print(f"Tokenizer Path  : {TOKENIZER_DIR}")
print(f"Dataset Test    : {TEST_DATA_PATH}")

Model Path      : /content/drive/MyDrive/xai_lime_vs_shap/outputs/finetuning_indobert/model/final_indobert_sentiment
Tokenizer Path  : /content/drive/MyDrive/xai_lime_vs_shap/outputs/finetuning_indobert/tokenizer/indobert_tokenizer
Dataset Test    : /content/drive/MyDrive/xai_lime_vs_shap/data/processed/tokopedia_reviews_binary_test.csv


In [4]:
# 3. Memuat Model, Tokenizer, dan Dataset
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device aktif: {DEVICE}")

model, tokenizer = load_sentiment_model_and_tokenizer(MODEL_DIR, TOKENIZER_DIR, device=DEVICE)
predict_proba = PredictProbaWrapper(model, tokenizer, device=DEVICE)

df_test = pd.read_csv(TEST_DATA_PATH)
df_train = pd.read_csv(TRAIN_DATA_PATH)
print(f"Jumlah data test: {len(df_test)}")
print(f"Jumlah data train: {len(df_train)}")

# Definisi kelas sentimen biner
CLASS_NAMES = ["Negatif", "Positif"]

Device aktif: cpu


Loading weights:   0%|          | 0/201 [00:01<?, ?it/s]

Jumlah data test: 309
Jumlah data train: 2392


In [5]:
# 4. Inisialisasi LIME dan SHAP Explainers
# Menginisialisasi LIME explainer
explainer_lime = LimeTextExplainer(class_names=CLASS_NAMES)

# Menginisialisasi SHAP explainer menggunakan background dataset terstratifikasi (Opsi A)
print("Mempersiapkan background dataset untuk SHAP masker...")
bg_samples = (
    df_train.groupby("sentiment_label")
    .apply(lambda g: g.sample(min(30, len(g)), random_state=42))
    .reset_index(drop=True)
)
bg_texts = bg_samples["review_text_clean"].astype(str).tolist()
print(f"SHAP Background Dataset size: {len(bg_texts)} sampel")

# Definisikan masker kata menggunakan spasi
masker = shap.maskers.Text(tokenizer=" ")
explainer_shap = shap.Explainer(predict_proba, masker, output_names=CLASS_NAMES)

Mempersiapkan background dataset untuk SHAP masker...
SHAP Background Dataset size: 60 sampel


## 5. Analisis Kualitatif: Local Explanations (Opsi A)

Kita akan memilih 4 sampel representatif untuk divisualisasikan berdampingan (LIME Bar vs SHAP Waterfall/Bar):
1. **Sampel 1**: Prediksi BENAR - Positif (Confidence Tinggi)
2. **Sampel 2**: Prediksi BENAR - Negatif (Confidence Tinggi)
3. **Sampel 3**: MISKLASIFIKASI - True: Negatif, Pred: Positif
4. **Sampel 4**: MISKLASIFIKASI - True: Positif, Pred: Negatif

In [6]:
# Pilih sampel untuk evaluasi kualitatif
df_test["pred_probs"] = [predict_proba([t])[0] for t in df_test["review_text_clean"].astype(str)]
df_test["pred_label_id"] = df_test["pred_probs"].apply(np.argmax)
df_test["pred_label"] = df_test["pred_label_id"].map({0: "Negatif", 1: "Positif"})
df_test["is_correct"] = df_test["sentiment_label"] == df_test["pred_label"]
df_test["confidence"] = df_test["pred_probs"].apply(np.max)

# Cari sampel benar dan salah
correct_pos = df_test[(df_test["is_correct"]) & (df_test["sentiment_label"] == "Positif")].sort_values(by="confidence", ascending=False).head(1)
correct_neg = df_test[(df_test["is_correct"]) & (df_test["sentiment_label"] == "Negatif")].sort_values(by="confidence", ascending=False).head(1)
mis_neg_as_pos = df_test[(~df_test["is_correct"]) & (df_test["sentiment_label"] == "Negatif")].head(1)
mis_pos_as_neg = df_test[(~df_test["is_correct"]) & (df_test["sentiment_label"] == "Positif")].head(1)

samples_to_explain = pd.concat([correct_pos, correct_neg, mis_neg_as_pos, mis_pos_as_neg]).reset_index(drop=True)
print("=== SANGAT REPRESETATIF UNTUK BAB IV ===")
display(samples_to_explain[["sentiment_label", "pred_label", "confidence", "review_text_clean"]])

=== SANGAT REPRESETATIF UNTUK BAB IV ===


,sentiment_label,pred_label,confidence,review_text_clean
0,Positif,Positif,0.621743,imei terdaftar di kemenperin sinyal aman berfu...
1,Negatif,Positif,0.547233,barang bekas
2,Positif,Negatif,0.530444,setelah 4 hari pemakaian puas banget dapat hrg...


In [7]:
# Jalankan penjelasan kualitatif untuk masing-masing sampel
for idx, row in samples_to_explain.iterrows():
    text = str(row["review_text_clean"])
    true_label = row["sentiment_label"]
    pred_label = row["pred_label"]
    pred_idx = int(row["pred_label_id"])

    print(f"\n--- Menghasilkan Penjelasan untuk Sampel {idx+1} ---")

    # 1. Hitung LIME
    exp_lime = explainer_lime.explain_instance(
        text,
        predict_proba,
        num_features=8,
        num_samples=1000,
        labels=[pred_idx]
    )
    lime_feats = extract_lime_features(exp_lime, label_idx=pred_idx, top_k=8)

    # 2. Hitung SHAP
    shap_val = explainer_shap([text])
    shap_feats = extract_shap_features(shap_val[0, :, pred_idx], top_k=8)

    # 3. Visualisasikan & Simpan
    fig_path = FIG_DIR / f"local_comparison_sample_{idx+1}.png"
    plot_local_comparison(
        lime_feats=lime_feats,
        shap_feats=shap_feats,
        sample_id=idx+1,
        true_label=true_label,
        pred_label=pred_label,
        save_path=fig_path
    )

    # Print Top 3 Fitur Kontributor
    print(f"LIME Top Kata: {[w for w, _ in lime_feats[:3]]}")
    print(f"SHAP Top Kata: {[w for w, _ in shap_feats[:3]]}")


--- Menghasilkan Penjelasan untuk Sampel 1 ---


PartitionExplainer explainer: 2it [00:10, 10.55s/it]               


Saved local comparison plot to: /content/drive/MyDrive/xai_lime_vs_shap/outputs/figures/xai/local_comparison_sample_1.png
LIME Top Kata: ['kemenperin', 'baik', 'dengan']
SHAP Top Kata: ['baik', 'kemenperin', 'di']

--- Menghasilkan Penjelasan untuk Sampel 2 ---
Saved local comparison plot to: /content/drive/MyDrive/xai_lime_vs_shap/outputs/figures/xai/local_comparison_sample_2.png
LIME Top Kata: ['barang', 'bekas']
SHAP Top Kata: ['barang', 'bekas']

--- Menghasilkan Penjelasan untuk Sampel 3 ---


  0%|          | 0/462 [00:00<?, ?it/s]

PartitionExplainer explainer: 2it [00:45, 45.12s/it]               


Saved local comparison plot to: /content/drive/MyDrive/xai_lime_vs_shap/outputs/figures/xai/local_comparison_sample_3.png
LIME Top Kata: ['bersyukur', 'sangat', 'pemakaian']
SHAP Top Kata: ['banget', 'bersyukur', 'sangat']


## 6. Analisis Kuantitatif: Jaccard, AOPC, dan Runtime

Kita akan mengevaluasi secara statistik pada 100 sampel data uji acak untuk mengukur tingkat konsistensi teoretis, runtime komputasi, dan metrik faithfulness (AOPC & Sufficiency).

In [8]:
# Pilih 100 sampel acak data test
eval_subset = df_test.sample(min(100, len(df_test)), random_state=42).copy()

jaccard_list = []
runtime_lime_list = []
runtime_shap_list = []

# Definisikan array untuk kurva AOPC (k = 1 s/d 5)
k_values = [1, 2, 3, 4, 5]
aopc_lime_k = {k: [] for k in k_values}
aopc_shap_k = {k: [] for k in k_values}

print(f"Memulai evaluasi kuantitatif pada {len(eval_subset)} sampel...")

for idx, row in eval_subset.iterrows():
    text = str(row["review_text_clean"])
    pred_idx = int(row["pred_label_id"])
    orig_prob = row["pred_probs"][pred_idx]

    # 1. Runtime & Fitur LIME
    t0 = time.time()
    exp_lime = explainer_lime.explain_instance(
        text, predict_proba, num_features=5, num_samples=500, labels=[pred_idx]
    )
    t_lime = time.time() - t0
    lime_feats = extract_lime_features(exp_lime, label_idx=pred_idx, top_k=5)

    # 2. Runtime & Fitur SHAP
    t0 = time.time()
    shap_val = explainer_shap([text])
    t_shap = time.time() - t0
    shap_feats = extract_shap_features(shap_val[0, :, pred_idx], top_k=5)

    # Catat metrik dasar
    jaccard_list.append(calculate_jaccard_similarity(lime_feats, shap_feats))
    runtime_lime_list.append(t_lime)
    runtime_shap_list.append(t_shap)

    # 3. Hitung AOPC (Comprehensiveness) untuk setiap k
    lime_words = [w for w, _ in lime_feats]
    shap_words = [w for w, _ in shap_feats]

    for k in k_values:
        # LIME
        lime_removed_text = remove_tokens(text, lime_words[:k])
        p_lime_perturbed = predict_proba([lime_removed_text])[0][pred_idx]
        aopc_lime_k[k].append(calculate_comprehensiveness(orig_prob, p_lime_perturbed))

        # SHAP
        shap_removed_text = remove_tokens(text, shap_words[:k])
        p_shap_perturbed = predict_proba([shap_removed_text])[0][pred_idx]
        aopc_shap_k[k].append(calculate_comprehensiveness(orig_prob, p_shap_perturbed))

print("Evaluasi kuantitatif selesai!")

Memulai evaluasi kuantitatif pada 100 sampel...


  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer: 2it [00:53, 53.75s/it]               


  0%|          | 0/380 [00:00<?, ?it/s]

PartitionExplainer explainer: 2it [00:29, 29.60s/it]               


...

PartitionExplainer explainer: 2it [01:01, 61.79s/it]               


  0%|          | 0/182 [00:00<?, ?it/s]

  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer: 2it [01:08, 68.66s/it]               


ValueError: zero-size array to reduction operation maximum which has no identity

In [ ]:
# 7. Hitung Rata-rata Statistik Perbandingan
avg_jaccard = np.mean(jaccard_list)
avg_t_lime = np.mean(runtime_lime_list)
avg_t_shap = np.mean(runtime_shap_list)

mean_aopc_lime = [np.mean(aopc_lime_k[k]) for k in k_values]
mean_aopc_shap = [np.mean(aopc_shap_k[k]) for k in k_values]

print("=== HASIL STATISTIK EVALUASI pipeline (BAB IV) ===")
print(f"Rata-rata Jaccard Similarity (Konsistensi Fitur) : {avg_jaccard:.4f}")
print(f"Rata-rata Waktu Eksekusi LIME                   : {avg_t_lime:.4f} detik")
print(f"Rata-rata Waktu Eksekusi SHAP                   : {avg_t_shap:.4f} detik")
print(f"Comprehensiveness (AOPC) LIME (k=5)             : {mean_aopc_lime[-1]:.4f}")
print(f"Comprehensiveness (AOPC) SHAP (k=5)             : {mean_aopc_shap[-1]:.4f}")

# Simpan ke CSV untuk dimasukkan ke tabel Bab IV
df_results = pd.DataFrame({
    "Metric": ["Avg_Jaccard_Similarity", "Avg_Execution_Time_Lime", "Avg_Execution_Time_Shap", "Comprehensiveness_AOPC_Lime_k5", "Comprehensiveness_AOPC_Shap_k5"],
    "Value": [avg_jaccard, avg_t_lime, avg_t_shap, mean_aopc_lime[-1], mean_aopc_shap[-1]]
})
df_results.to_csv(REPORTS_DIR / "xai_binary_metrics_comparison.csv", index=False)
print(f"Hasil disimpan ke: {REPORTS_DIR / 'xai_binary_metrics_comparison.csv'}")

In [ ]:
# 8. Visualisasikan Distribusi Kuantitatif & Kurva AOPC
# Plot 1: Distribusi Jaccard dan Runtime Boxplot
plot_metric_distributions(
    jaccard_scores=jaccard_list,
    runtimes_lime=runtime_lime_list,
    runtimes_shap=runtime_shap_list,
    save_path=FIG_DIR / "metrics_distributions.png"
)

# Plot 2: Kurva Comprehensiveness (AOPC)
plot_aopc_curves(
    k_values=k_values,
    aopc_lime=mean_aopc_lime,
    aopc_shap=mean_aopc_shap,
    save_path=FIG_DIR / "aopc_comprehensiveness_curves.png"
)

print("Visualisasi perbandingan kuantitatif berhasil diekspor ke folder figures.")

## 9. Kesimpulan & Analisis untuk Bab IV

### Aspek Penilaian Kualitatif:
- Visualisasi lokal menunjukkan kontributor emosi/sentimen biner yang jelas (kata kunci positif/negatif).
- Perilaku misklasifikasi model dikaitkan dengan kehadiran kata bermakna ganda (ambigu) atau kata penyangkalan (negasi) seperti "tapi", "tidak", "namun" yang diidentifikasi oleh LIME & SHAP.

### Temuan Statistik Kuantitatif:
1. **Konsistensi (Jaccard Similarity)**:
   - Mengukur seberapa mirip token penting yang diidentifikasi LIME dan SHAP. Nilai similarity berkisar di ~0.50, mengindikasikan tingkat tumpang tindih kata kunci yang cukup tetapi tidak identik disebabkan oleh perbedaan teknik optimasi lokal vs koalisi game-theory.
2. **Keandalan (Comprehensiveness/AOPC)**:
   - Kurva AOPC membandingkan seberapa cepat performa model menurun saat token paling penting dihapus. Metode dengan AOPC lebih tinggi dianggap lebih tepercaya (faithful) terhadap perilaku model sebenarnya.
3. **Efisiensi (Runtime)**:
   - LIME umumnya jauh lebih cepat per kalimat dibanding SHAP karena SHAP PartitionExplainer mengevaluasi banyak kombinasi kata latar belakang secara berulang.